# Habits: per-move features from engine analysis

**Milestone:** "Estimate player habits from game history" (README Teaching). Gambit estimates a
player's level from **habits** (recurring mistakes, missed tactics, risky or passive tendencies)
rather than from results.

For every move the player made we ask the engine (a full-strength *analyst*):

* **Centipawn loss (CPL)**: eval of the best move minus eval of the played move, both from the
  player's point of view, computed at the same depth (`root_moves=[played]`), capped at `cpl_cap`.
* **Classification**: inaccuracy / mistake / blunder by CPL thresholds in game.yaml
  (`teaching.cpl_thresholds`, lichess-like defaults, to be tuned).
* **Missed forcing move**: the best move was a capture or check and the player did not find it
  (and lost at least an inaccuracy's worth). Specific tactic detectors (fork, pin, ...) are TODO.
* **Phase**: opening / middlegame / endgame (config thresholds).
* **Time use**: think time from the PGN `[%emt]` annotation (our logger writes it) or the robot log.

Per player these aggregate into a `HabitProfile`.

**TODO checklist**
- [ ] CPL and classification (`centipawn_loss`, `classify_cpl`).
- [ ] Phase (`non_pawn_material`, `game_phase`).
- [ ] Per-move analysis (`analyse_move`, `analyse_game`, `player_color_in_game`).
- [ ] Profile aggregation (`build_habit_profile`).
- [ ] Self-checks (plan at the bottom).
- [ ] Tactic detectors (`TACTIC_DETECTORS`): fork, pin/skewer, hanging piece, back-rank.
- [ ] Risky vs passive tendencies (README): e.g. share of moves that reduce king safety, trade
      offers declined, pawn-storm frequency. Define, then validate they are stable per player.
- [ ] Use win-probability loss instead of raw CPL in lopsided positions (CPL of 300 at +900
      barely matters); lichess uses a logistic transform.

In [ ]:
%run ../../gambit_game/notebooks/engine.ipynb

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from typing import Callable, Iterable, Optional

import chess
import chess.pgn
import pandas as pd

TEACH_CFG = GAME_CFG["teaching"]
CPL_THRESHOLDS = TEACH_CFG["cpl_thresholds"]
CLASSES = ("best", "good", "inaccuracy", "mistake", "blunder")

## CPL and classification

In [ ]:
def centipawn_loss(best_cp: int, played_cp: int, cap: int = TEACH_CFG["cpl_cap"]) -> int:
    """Both scores from the mover's POV. Never negative (the engine's best can be underestimated)."""
    # Design: difference best - played, clipped to 0..cap. The cap stops one mate-score blunder
    # (scores near mate_score_cp) from dominating a whole game's average.
    # TODO: implement
    raise NotImplementedError


def classify_cpl(cpl: int, is_best: bool = False, thresholds: dict = CPL_THRESHOLDS) -> str:
    """One of CLASSES for a move's CPL."""
    # Design: "best" if it was the engine's move (or lost nothing); otherwise compare with the
    # blunder / mistake / inaccuracy thresholds from the most severe down; below all -> "good".
    # TODO: implement
    raise NotImplementedError

## Game phase

Simple and explainable: opening by move number, endgame by remaining non-pawn material
(N = B = 3, R = 5, Q = 9, both sides together), middlegame otherwise.

In [ ]:
PHASE_VALUE = {chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9}


def non_pawn_material(board: chess.Board) -> int:
    """Sum of PHASE_VALUE over all pieces of both colours."""
    # TODO: implement
    raise NotImplementedError


def game_phase(board: chess.Board, cfg: dict = TEACH_CFG["phase"]) -> str:
    """'opening' | 'middlegame' | 'endgame'."""
    # Design: endgame check first (low material wins even early in the game), then opening by
    # board.fullmove_number, else middlegame. Thresholds from cfg.
    # TODO: implement
    raise NotImplementedError

## Tactic detectors (TODO)

Each detector gets the position before the move, the engine's best line and the move played, and
returns `True` if the player missed that tactic. They plug into `analyse_move(detectors=...)`.

In [ ]:
def detect_missed_fork(board: chess.Board, best: Candidate, played: chess.Move) -> bool:
    """Did the player miss a fork the engine's best line had?"""
    # Design hints: best move (or the PV's 2nd own move) attacks two+ valuable enemy pieces at once
    # (board.attacks after pushing the move), and the played move did not.
    # TODO: implement
    raise NotImplementedError


def detect_missed_pin(board: chess.Board, best: Candidate, played: chess.Move) -> bool:
    """Did the player miss a pin / skewer?"""
    # Design hints: best move creates a pin/skewer (board.is_pinned on the opponent after the move).
    # TODO: implement
    raise NotImplementedError


def detect_hanging_piece(board: chess.Board, best: Candidate, played: chess.Move) -> bool:
    """Did the played move leave a piece hanging?"""
    # Design hints: after the played move, one of the player's pieces is attacked and not defended
    # (or defended by less than its value) and the engine's reply captures it.
    # TODO: implement
    raise NotImplementedError


TACTIC_DETECTORS: dict[str, Callable] = {
    "fork": detect_missed_fork,
    "pin": detect_missed_pin,
    "hanging_piece": detect_hanging_piece,
}

## Per-move analysis

In [ ]:
@dataclass
class MoveAnalysis:
    game_id: str
    ply: int
    color: str                        # "white" | "black" (the player who moved)
    fen: str                          # position before the move
    uci: str
    best_uci: str
    best_cp: int                      # mover POV
    played_cp: int                    # mover POV
    cpl: int
    classification: str
    phase: str
    best_was_forcing: bool            # best move is a capture or check
    missed_forcing: bool
    think_time_s: Optional[float] = None
    tactics_missed: list = field(default_factory=list)


def analyse_move(board: chess.Board, move: chess.Move, analyst: EngineLike, game_id: str = "",
                 think_time_s: Optional[float] = None, multipv: int = 1,
                 detectors: Iterable[str] = ()) -> MoveAnalysis:
    """Grade one move: CPL, classification, phase, forcing-move miss, tactics."""
    # Design:
    # - analyst.analyse(board, multipv) for the best line. If the played move is not among the
    #   candidates, score it with a second call restricted by root_moves=[move], so both scores
    #   come from the same search depth (otherwise CPL is noise).
    # - Treat the move as best if it is the engine's move or scores at least as well (the engine
    #   is not perfect).
    # - Forcing = best move is a capture or gives check (board.is_capture / board.gives_check);
    #   missed_forcing = forcing, not best, and CPL at least the inaccuracy threshold.
    # - Run the named TACTIC_DETECTORS; fill MoveAnalysis (colour of board.turn, board.ply(), FEN).
    # TODO: implement
    raise NotImplementedError


def player_color_in_game(game: chess.pgn.Game, player: str) -> Optional[chess.Color]:
    """Which colour `player` (case-insensitive name) played in a PGN game, or None."""
    # Design: compare with the White / Black headers, ignoring case.
    # TODO: implement
    raise NotImplementedError


def analyse_game(game: chess.pgn.Game, analyst: EngineLike, color: chess.Color,
                 game_id: Optional[str] = None, multipv: int = 1,
                 detectors: Iterable[str] = ()) -> list[MoveAnalysis]:
    """Analyse every move `color` made in a PGN game. Think time from `[%emt]` if present."""
    # Design:
    # - game_id default from the Site / Event headers (lichess puts the game URL in Site).
    # - Walk game.mainline() from game.board(), analysing only `color`'s moves; node.emt() gives
    #   the think time when the PGN has clock annotations.
    # TODO: implement
    raise NotImplementedError

## HabitProfile: aggregate over many games

In [ ]:
@dataclass
class HabitProfile:
    player: str
    n_games: int
    n_moves: int
    acpl: float
    per_game_acpl: list
    class_rates: dict                 # classification -> share of moves
    acpl_by_phase: dict               # phase -> ACPL
    moves_by_phase: dict
    missed_forcing_rate: float        # missed forcing moves / positions where best was forcing
    n_forcing_positions: int
    mean_think_time_s: Optional[float]

    def to_dict(self) -> dict:
        """Plain dict (dataclasses.asdict) for JSON / tables."""
        # TODO: implement
        raise NotImplementedError


def analyses_to_frame(analyses: list[MoveAnalysis]) -> pd.DataFrame:
    """One row per MoveAnalysis."""
    # TODO: implement
    raise NotImplementedError


def build_habit_profile(player: str, analyses: list[MoveAnalysis]) -> HabitProfile:
    """Aggregate per-move analyses into a HabitProfile."""
    # Design:
    # - Use a DataFrame (analyses_to_frame) and groupby: per-game ACPL, ACPL and move count per
    #   phase, share of each classification in CLASSES.
    # - missed_forcing_rate is over positions where the best move was forcing (not all moves),
    #   otherwise it mostly measures how tactical the games were.
    # - Empty input -> a profile of NaNs / zeros instead of an exception.
    # - Mean think time only over moves that have one.
    # TODO: implement
    raise NotImplementedError

## Self-check (MaterialEngine as a stand-in analyst; real analysis uses Stockfish)

In [ ]:
import io

# Self-check plan (write these as asserts once the functions above work):
# - centipawn_loss(50, -30) == 80; (10, 40) -> 0; (9000, -9000) -> cpl_cap.
# - classify_cpl over 0, 20, 60, 150, 400 -> best, good, inaccuracy, mistake, blunder (default thresholds).
# - game_phase: start -> opening; "4k3/8/8/8/8/8/4P3/R3K3 w - - 0 40" -> endgame; a full-material
#   position at move 15 -> middlegame.
# - Scholar's mate PGN ('[White "alice"]\n[Black "bob"]\n\n1. e4 {[%emt 0:00:04]} e5 2. Qh5 Nc6
#   3. Bc4 Nf6 4. Qxf7# 1-0') with MaterialEngine(depth=2) as analyst:
#   * player_color_in_game is case-insensitive ("BOB" -> black);
#   * black's moves e7e5, b8c6, g8f6; the last is a blunder at cpl_cap;
#   * white's Qxf7# is "best" and best_was_forcing; white's first think time is 4.0 s.
# - build_habit_profile("bob", ...): 3 moves, 1 game, blunder share 1/3, acpl = mean CPL.
#   (player_model.ipynb's self-check reuses black's analyses: keep them in a variable `black`.)
# Example shape of a test:
# assert centipawn_loss(50, -30) == 80
print("Self-check: not written yet (see plan above)")